## Phase 1 – Prepare Structured Data

In [3]:
import os

# Create sample datasets

# Customer Data
with open('customers.csv', 'w') as f:
    f.write('cust_id,name,city\n')
    f.write('1,Alice,New York\n')
    f.write('2,Bob,Los Angeles\n')
    f.write('3,Charlie,Chicago\n')
    f.write('4,Diana,Houston\n')

# Transactions Data
with open('transactions.csv', 'w') as f:
    f.write('txn_id,cust_id,amount,category\n')
    f.write('101,1,250,Electronics\n')
    f.write('102,2,100,Books\n')
    f.write('103,1,300,Fashion\n')
    f.write('104,3,150,Electronics\n')
    f.write('105,4,200,Books\n')

print('customers.csv created:')
!cat customers.csv

print('\ntransactions.csv created:')
!cat transactions.csv

customers.csv created:
cust_id,name,city
1,Alice,New York
2,Bob,Los Angeles
3,Charlie,Chicago
4,Diana,Houston

transactions.csv created:
txn_id,cust_id,amount,category
101,1,250,Electronics
102,2,100,Books
103,1,300,Fashion
104,3,150,Electronics
105,4,200,Books


## Phase 2 – Load Data into DataFrames

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SparkSQLLab").getOrCreate()

# Load and infer schema automatically
custDF = spark.read.option("header", True).option("inferSchema", True).csv("customers.csv")
txnDF = spark.read.option("header", True).option("inferSchema", True).csv("transactions.csv")

print("Customer DataFrame:")
custDF.show()

print("Transaction DataFrame:")
txnDF.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/04 08:28:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/04 08:28:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/04 08:28:47 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
                                                                                

Customer DataFrame:
+-------+-------+-----------+
|cust_id|   name|       city|
+-------+-------+-----------+
|      1|  Alice|   New York|
|      2|    Bob|Los Angeles|
|      3|Charlie|    Chicago|
|      4|  Diana|    Houston|
+-------+-------+-----------+

Transaction DataFrame:
+------+-------+------+-----------+
|txn_id|cust_id|amount|   category|
+------+-------+------+-----------+
|   101|      1|   250|Electronics|
|   102|      2|   100|      Books|
|   103|      1|   300|    Fashion|
|   104|      3|   150|Electronics|
|   105|      4|   200|      Books|
+------+-------+------+-----------+



## Phase 3 – Register Temporary Views

In [5]:
# Register DataFrames as SQL views
custDF.createOrReplaceTempView("customers")
txnDF.createOrReplaceTempView("transactions")

print("Temporary views 'customers' and 'transactions' registered.")

Temporary views 'customers' and 'transactions' registered.


## Phase 4 – Run Basic SQL Queries

In [6]:
# Query top customers
print("SELECT * FROM customers:")
spark.sql("SELECT * FROM customers").show()

# Retrieve specific columns
print("SELECT name, city FROM customers:")
spark.sql("SELECT name, city FROM customers").show()

# Add filters
print("SELECT * FROM customers WHERE city='New York':")
spark.sql("SELECT * FROM customers WHERE city='New York'").show()

SELECT * FROM customers:
+-------+-------+-----------+
|cust_id|   name|       city|
+-------+-------+-----------+
|      1|  Alice|   New York|
|      2|    Bob|Los Angeles|
|      3|Charlie|    Chicago|
|      4|  Diana|    Houston|
+-------+-------+-----------+

SELECT name, city FROM customers:
+-------+-----------+
|   name|       city|
+-------+-----------+
|  Alice|   New York|
|    Bob|Los Angeles|
|Charlie|    Chicago|
|  Diana|    Houston|
+-------+-----------+

SELECT * FROM customers WHERE city='New York':
+-------+-----+--------+
|cust_id| name|    city|
+-------+-----+--------+
|      1|Alice|New York|
+-------+-----+--------+



## Phase 5 – Perform Aggregations

In [7]:
# Find total transaction amount by category
print("Total transaction amount by category:")
spark.sql("""
SELECT category, SUM(amount) AS total_sales
FROM transactions
GROUP BY category
ORDER BY total_sales DESC
""").show()

Total transaction amount by category:
+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|        400|
|    Fashion|        300|
|      Books|        300|
+-----------+-----------+



## Phase 6 – Use Joins Between Tables

In [8]:
# Join customers and transactions to find purchase history
print("Purchase history by customer:")
spark.sql("""
SELECT c.name, c.city, t.category, t.amount
FROM customers c
JOIN transactions t
ON c.cust_id = t.cust_id
ORDER BY t.amount DESC
""").show()

Purchase history by customer:
+-------+-----------+-----------+------+
|   name|       city|   category|amount|
+-------+-----------+-----------+------+
|  Alice|   New York|    Fashion|   300|
|  Alice|   New York|Electronics|   250|
|  Diana|    Houston|      Books|   200|
|Charlie|    Chicago|Electronics|   150|
|    Bob|Los Angeles|      Books|   100|
+-------+-----------+-----------+------+



## Phase 7 – Apply Conditional Logic and Derived Columns

In [9]:
# Use CASE WHEN for derived attributes
print("Transactions with derived 'txn_type':")
spark.sql("""
SELECT name, amount,
CASE
    WHEN amount > 200 THEN 'High-Value'
    WHEN amount BETWEEN 100 AND 200 THEN 'Medium-Value'
    ELSE 'Low-Value'
END AS txn_type
FROM customers c
JOIN transactions t ON c.cust_id = t.cust_id
""").show()

Transactions with derived 'txn_type':
+-------+------+------------+
|   name|amount|    txn_type|
+-------+------+------------+
|  Alice|   300|  High-Value|
|  Alice|   250|  High-Value|
|    Bob|   100|Medium-Value|
|Charlie|   150|Medium-Value|
|  Diana|   200|Medium-Value|
+-------+------+------------+



## Phase 8 – Use SQL Functions

In [10]:
# Compute averages and distinct counts
print("Category-wise transaction counts and average amounts:")
spark.sql("""
SELECT category,
       COUNT(*) AS total_txns,
       AVG(amount) AS avg_amount
FROM transactions
GROUP BY category
""").show()

Category-wise transaction counts and average amounts:
+-----------+----------+----------+
|   category|total_txns|avg_amount|
+-----------+----------+----------+
|    Fashion|         1|     300.0|
|Electronics|         2|     200.0|
|      Books|         2|     150.0|
+-----------+----------+----------+



## Phase 9 – Create Global Views (Optional)

In [11]:
# Global views persist across Spark sessions
custDF.createOrReplaceGlobalTempView("global_customers")
print("Querying global_temp.global_customers:")
spark.sql("SELECT * FROM global_temp.global_customers").show()

# Note: In a true multi-session environment, you would start a new SparkSession
# to confirm its persistence. For this notebook, we just demonstrate creation and query.

Querying global_temp.global_customers:
+-------+-------+-----------+
|cust_id|   name|       city|
+-------+-------+-----------+
|      1|  Alice|   New York|
|      2|    Bob|Los Angeles|
|      3|Charlie|    Chicago|
|      4|  Diana|    Houston|
+-------+-------+-----------+



## Phase 10 – Cache Query Results

In [12]:
# Cache a view for faster repeated queries
spark.catalog.cacheTable("transactions")
print("Caching 'transactions' table.")

# Query the cached table
print("Count from cached 'transactions' table:")
spark.sql("SELECT COUNT(*) FROM transactions").show()

# Uncache the table
spark.catalog.uncacheTable("transactions")
print("'transactions' table uncached.")

Caching 'transactions' table.
Count from cached 'transactions' table:
+--------+
|count(1)|
+--------+
|       5|
+--------+

'transactions' table uncached.
